In [34]:
from dotenv import load_dotenv

load_dotenv()

True

## Document Scrappers

### LLama Parser

In [1]:
# advanced script: https://github.com/run-llama/llama_parse/blob/main/examples/demo_advanced.ipynb

import nest_asyncio
from llama_parse import LlamaParse
from llama_index.core import SimpleDirectoryReader


nest_asyncio.apply()

In [3]:
parser = LlamaParse(
    # api_key="",  # can also be set in your env as LLAMA_CLOUD_API_KEY
    result_type="text",  # "markdown" and "text" are available
    num_workers=4,  # if multiple files passed, split in `num_workers` API calls
    verbose=True,
    language="en",  # Optionally you can define a language, default=en
)

# # sync
# documents = parser.load_data("./my_file.pdf")

# sync batch
# documents = parser.load_data(["./documents/13-0652.pdf", "./documents/Blouin_2012.pdf", "./documents/Guyeux_2024.pdf"])

# # async
# documents = await parser.aload_data("./my_file.pdf")

# # async batch
# documents = await parser.aload_data(["./my_file1.pdf", "./my_file2.pdf"])

file_extractor = {".pdf": parser}
documents = SimpleDirectoryReader(
    "./documents", file_extractor=file_extractor
).load_data()

Started parsing the file under job_id 8aaefe1e-f8b7-4f1b-bae8-8a53e7cdbb51
.............Started parsing the file under job_id e25cfaba-210c-4156-89bf-c61a50cb8d29
...........Started parsing the file under job_id 74913510-f4b6-45e0-822d-60d252379525
............

In [6]:
print(documents[0].text[:1000])

 Progenitor “Mycobacterium canettii”
    Clone Responsible for Lymph Node
           Tuberculosis Epidemic, Djibouti
                    Yann Blouin, Géraldine Cazajous, Céline Dehan, Charles Soler, Rithy Vong,



             Mohamed Osman Hassan, Yolande Hauck, Christian Boulais, Dina Andriamanantena,
                   Christophe Martinaud, Émilie Martin, Christine Pourcel, and Gilles Vergnaud



      “Mycobacterium        canettii,”   an   opportunistic     human       higher among expatriate than among Djiboutian patients
pathogen living in an unknown environmental reservoir, is                   and that patients with M. canettii infection were significant-
the progenitor species from which Mycobacterium tubercu-                    ly younger than those with          M. tuberculosis infection (2).
losis emerged. Since its discovery in 1969, most of the ≈70                 These findings suggested that the Djiboutian popula-
known    M. canettii strains were isolated in the Repub

### OpenAi - File Search

In [39]:
from openai import OpenAI
client = OpenAI()
 
assistant = client.beta.assistants.create(
  
  name="Medical Assistant",
  instructions="You are an expert medical analyst. Use you knowledge base to answer questions about medical research papers.",
  model="gpt-4-turbo",
  # model="gpt-3.5-turbo-instruct",
  tools=[{"type": "file_search"}],
)

BadRequestError: Error code: 400 - {'error': {'message': "Invalid value: 'file_search'. Supported values are: 'code_interpreter', 'function', and 'retrieval'.", 'type': 'invalid_request_error', 'param': 'tools[0].type', 'code': 'invalid_value'}}

In [ ]:
# Create a vector store caled "Medical Documents"
vector_store = client.beta.vector_stores.create(name="Medical Documents")
 
# Ready the files for upload to OpenAI
file_paths = ["documents/13-0652.pdf", "documents/Blouin_2012.pdf", "documents/Guyeux_2024.pdf"]
file_streams = [open(path, "rb") for path in file_paths]
 
# Use the upload and poll SDK helper to upload the files, add them to the vector store,
# and poll the status of the file batch for completion.
file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

In [ ]:
# You can print the status and the file counts of the batch to see the result of this operation.
print(file_batch.status)
print(file_batch.file_counts)

### LangChain - PyPdf Loader

In [11]:
# code reference https://python.langchain.com/v0.1/docs/modules/data_connection/document_loaders/pdf/
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()
loader = PyPDFLoader("documents/13-0652.pdf")
pages = loader.load_and_split()
faiss_index = FAISS.from_documents(pages, OpenAIEmbeddings())

In [16]:
docs = faiss_index.similarity_search("Où trouve-t-on principalement des M.canettii ?", k=2)

In [17]:
for doc in docs:
    print(str(doc.metadata["page"]) + ":", doc.page_content[:1000])

6: “M. canettii” Epidemic, Djibouti
that different M. tuberculosis lineages emerged indepen-
dently from its reservoir. These different lineages might be distinguished by traces of ancestral horizontal gene transfer events, visible in the very internal branches of M. tuberculosis evolution, as observed here within clone A. We could not identify any such fossils of early hori-zontal gene transfer events, which is compatible with a model in which the most recent ancestor of M. tubercu-losis never lived in the environment. One possibility is that it acquired a key feature leading to speciation during the colonization of its human host, after infection from the environment. Another possibility is that the most re-cent ancestor of M. tuberculosis does not coincide with the speciation of M. tuberculosis the obligatory human pathogen (23) as suggested here by comparing evolu-tionary rates toward M. canettii clone A and toward M. 
tuberculosis. Clone A may mimic an earlier phase before M. tube

## Web Scrappers

### LangChain - FireCrawl

In [31]:
from langchain_community.document_loaders import FireCrawlLoader
loader = FireCrawlLoader(
     url="https://python.langchain.com/v0.1/docs/integrations/tools/google_search/", mode="crawl"
)

In [32]:
docs = loader.load()

ChunkedEncodingError: ('Connection broken: IncompleteRead(5061 bytes read, 3131 more expected)', IncompleteRead(5061 bytes read, 3131 more expected))

In [ ]:
docs

In [35]:
from firecrawl import FirecrawlApp

app = FirecrawlApp()
query = 'how to compile a python code?'
search_result = app.search(query)

In [38]:
for result in search_result:
    print(result)

{'content': '*   [Ubuntu](http://www.ubuntu.com)\n    \n*   [Community](http://community.ubuntu.com/)\n    \n*   [Ask!](http://askubuntu.com)\n    \n*   [Developer](http://developer.ubuntu.com)\n    \n*   [Design](http://design.ubuntu.com)\n    \n*   [Hardware](http://www.ubuntu.com/certification)\n    \n*   [Insights](http://insights.ubuntu.com/)\n    \n*   [Juju](https://jujucharms.com/)\n    \n*   [Shop](http://shop.ubuntu.com)\n    \n*   [More ›](#)\n    *   [Apps](http://apps.ubuntu.com)\n        \n    *   [Help](https://help.ubuntu.com)\n        \n    *   [Forum](http://ubuntuforums.org)\n        \n    *   [Launchpad](http://www.launchpad.net)\n        \n    *   [MAAS](http://maas.ubuntu.com)\n        \n    *   [Canonical](http://www.canonical.com)\n        \n\nNew\n\n**Stack Overflow Jobs** powered by Indeed: A job site that puts thousands of tech jobs at your fingertips (U.S. only). [Search jobs](/jobs?source=so-banner)\n\n[](# "dismiss")\n\n**Teams**\n\nQ&A for work\n\nConnect

In [36]:
FireCrawlLoader(
     url="https://python.langchain.com/v0.1/docs/integrations/tools/google_search/", mode="crawl"
)

### ScrapeGraphAI

### LangChain - Google Search

In [25]:
from langchain_community.utilities import GoogleSearchAPIWrapper
from langchain_core.tools import Tool

search = GoogleSearchAPIWrapper()

tool = Tool(
    name="google_search",
    description="Search Google for recent results.",
    func=search.run,
)

In [28]:
tool.run("how to compile a python code?")

'Jul 26, 2013 ... 5 Answers 5 ... Adding to Bryan\'s answer, if you simply want to compile a file or a bunch of files from a terminal, the py_compile module can be\xa0... Dec 24, 2009 ... python is an interpreted language, so you don\'t need to compile your scripts to make them run. The easiest way to get one running is to navigate\xa0... Dec 9, 2022 ... Python code is compiled to a bytecode that\'s stored in .pyc files. You can\'t compile it to a .exe file, but you can cheat a bit by taking\xa0... Jun 23, 2023 ... How do I "compile" my python files into a single executable app · Use PyInstaller to bundle your Python files and a copy of the Python runtime\xa0... Jul 25, 2023 ... Hi, how can I compile the python files to an executable binary file like Linux C and Golang in different platforms such as Linux,\xa0... Feb 1, 2022 ... You can also automatically compile all Python files using the compileall module. You can do it from the shell prompt by running compileall.py\xa0... These inst

### LangChain - SerpAPIWrapper